# Локалізація артефактів українською мовою

Цей ноутбук перегенеровує всі таблиці та рисунки з українськими підписами, заголовками
та підписами осей. Вхідні дані — проміжні файли з `results/thesis/`, що створені
англомовними ноутбуками 01–05. Вихід: `results/thesis/ua/`.

In [ ]:
import sys
import os

os.chdir(os.path.join(os.path.dirname(os.path.abspath("__file__")), ".."))
sys.path.insert(0, ".")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from pathlib import Path
from scipy import stats
from statsmodels.stats.multitest import fdrcorrection

from abpm_hemodynamic_coupling.config import Config, Columns
from analysis.thesis_stats import (
    rank_biserial, bootstrap_ci, bootstrap_corr_ci,
    format_median_iqr, format_mean_sd, derive_dipper_category,
    export_table, compute_icc1, variance_decomposition,
    format_apa_wilcoxon, format_apa_mannwhitney, format_apa_spearman,
    format_apa_friedman, format_apa_kruskal,
)

config = Config()
THESIS_DIR = Path("results/thesis")
UA_DIR = THESIS_DIR / "ua"
UA_DIR.mkdir(parents=True, exist_ok=True)
DPI = 300

# --- Ukrainian font support ---
matplotlib.rcParams["font.family"] = "DejaVu Sans"
sns.set_theme(style="whitegrid", font_scale=1.1)

print(f"Output directory: {UA_DIR}")

In [ ]:
# ===================================================================
# Master translation dictionary
# ===================================================================

# Context labels
CTX_UA = {
    "Baseline": "Базовий рівень",
    "Sleep": "Сон",
    "Before Testing": "Перед тестуванням",
    "Cognitive Pre": "Перед когн. навант.",
    "Cognitive Task": "Когнітивне навант.",
    "Physical Pre": "Перед фіз. навант.",
    "Physical Task": "Фізичне навант.",
    "Break": "Перерва",
    "Air Alert": "Повітряна тривога",
    "Manual Trigger": "Ручний запуск",
}

# Outcome variable labels
VAR_UA = {
    "SBP": "САТ",
    "DBP": "ДАТ",
    "HR": "ЧСС",
}

UNITS_UA = {
    "SBP": "мм рт. ст.",
    "DBP": "мм рт. ст.",
    "HR": "уд/хв",
    "САТ": "мм рт. ст.",
    "ДАТ": "мм рт. ст.",
    "ЧСС": "уд/хв",
}

# Table 1 variable names
TABLE1_VARS_UA = {
    "Age, years": "Вік, роки",
    "Height, cm": "Зріст, см",
    "Weight, kg": "Маса тіла, кг",
    "BMI, kg/m\u00b2": "ІМТ, кг/м\u00b2",
    "Monitoring duration, min": "Тривалість моніторування, хв",
    "Sleep duration, h": "Тривалість сну, год",
    "BP load, %": "Навантаження АТ, %",
    "SBP dip, %": "Нічне зниження САТ, %",
    "DBP dip, %": "Нічне зниження ДАТ, %",
    "Female sex": "Жіноча стать",
    "Young (\u226430 y)": "Молоді (\u226430 р.)",
    "Normal BMI": "Нормальний ІМТ",
    "Healthy diagnosis": "Здоровий діагноз моніторування",
    "Chronic diseases": "Хронічні захворювання",
    "Normal SBP dipper": "Нормальний діппер за САТ",
    "Normal DBP dipper": "Нормальний діппер за ДАТ",
    "Air alert during monitoring": "Повітряна тривога під час моніторування",
    "Sleep <7 h": "Сон <7 год",
    "Satisfied with sleep": "Задоволені сном",
    "Works under stress": "Працює в умовах стресу",
    "Mental labor": "Розумова праця",
    "University degree": "Вища освіта",
    "Registered partnership": "Зареєстроване партнерство",
    "Not smoking": "Не палить",
    "Drinks coffee": "Вживає каву",
    "Good food quality": "Якісне харчування",
    "Sleep before midnight": "Сон до опівночі",
}

# Dipping categories
DIPPER_UA = {
    "Reverse dipper": "Зворотний діппер",
    "Non-dipper": "Нон-діппер",
    "Normal dipper": "Нормальний діппер",
    "Extreme dipper": "Екстремальний діппер",
}

DIPPER_DEFS_UA = {
    "Зворотний діппер": "Зниження < 0%",
    "Нон-діппер": "0% \u2264 Зниження < 10%",
    "Нормальний діппер": "10% \u2264 Зниження < 20%",
    "Екстремальний діппер": "Зниження \u2265 20%",
}

# Direction translation
DIR_UA = {"increase": "підвищення", "decrease": "зниження"}

# Test names
TEST_UA = {
    "Wilcoxon signed-rank": "Критерій Вілкоксона",
    "Mann-Whitney U": "Критерій Манна-Вітні",
    "Friedman": "Критерій Фрідмана",
    "Kruskal-Wallis": "Критерій Краскела-Уолліса",
}

# Family names
FAMILY_UA = {
    "Paired (Wilcoxon)": "Парні (Вілкоксон)",
    "Group (Mann-Whitney)": "Групові (Манн-Вітні)",
    "Multi-condition (Friedman)": "Множинні умови (Фрідман)",
    "Dipping (Kruskal-Wallis)": "Діппінг (Краскел-Уолліс)",
}

# Group variable names
GROUP_UA = {
    "is_female": "Стать (жін.)",
    "is_young": "Вік (\u226430 р.)",
    "monitoring_diagnosis_is_healthy": "Діагноз моніт.",
    "sleep_less_than_7h": "Сон <7 год",
    "working_under_stress": "Стрес на роботі",
}

# Outcome column names for group comparisons
OUTCOME_UA = {
    "SBP_all_med": "Медіана САТ",
    "DBP_all_med": "Медіана ДАТ",
    "HR_all_med": "Медіана ЧСС",
    "SBP_all_iqr": "IQR САТ",
    "DBP_all_iqr": "IQR ДАТ",
    "HR_all_iqr": "IQR ЧСС",
}

def t(key, mapping):
    """Translate key using mapping, return original if not found."""
    return mapping.get(key, key)

print(f"Translation dicts loaded: {len(CTX_UA)} contexts, {len(TABLE1_VARS_UA)} T1 vars")

In [ ]:
# Load intermediate data
monitoring = pd.read_parquet(THESIS_DIR / "monitoring_labeled.parquet")
ctx_medians = pd.read_csv(THESIS_DIR / "subject_context_medians.csv")
agg = pd.read_csv(THESIS_DIR / "aggregated_clean.csv")
res = pd.read_csv(config.get_results_path(config.SUBJECT_METRICS_OUTPUT))
merged = agg.merge(res, on="participant_id", how="inner")

# Pivot for paired comparisons
sc_wide = ctx_medians.pivot_table(
    index="participant_id", columns="label",
    values=["SBP_med", "DBP_med", "HR_med"],
)
sc_wide.columns = [f"{val}_{lab}" for val, lab in sc_wide.columns]
sc_wide = sc_wide.reset_index()

print(f"Monitoring: {monitoring.shape}")
print(f"Context medians: {ctx_medians.shape}")
print(f"Aggregated: {agg.shape}")
print(f"Merged: {merged.shape}")

## Таблиці

In [ ]:
# --- Table 1: Cohort demographics (UA) ---
t1 = pd.read_csv(THESIS_DIR / "table_01_cohort_demographics.csv")
t1["Variable"] = t1["Variable"].map(TABLE1_VARS_UA).fillna(t1["Variable"])
t1 = t1.rename(columns={
    "Variable": "Змінна",
    "Overall (N=28)": "Загалом (N=28)",
    "Male (n=16)": "Чоловіки (n=16)",
    "Female (n=12)": "Жінки (n=12)",
    "Missing": "Пропуски",
})
export_table(t1, UA_DIR / "table_01_cohort_demographics")
display(t1)
print("Exported table_01 (UA)")

In [ ]:
# --- Table 2: BP by context (UA) ---
t2 = pd.read_csv(THESIS_DIR / "table_02_bp_by_context.csv")
t2["Context"] = t2["Context"].map(CTX_UA).fillna(t2["Context"])
t2 = t2.rename(columns={
    "Context": "Контекст",
    "Subjects": "Учасники",
    "Readings": "Вимірювання",
    "SBP, mmHg": "САТ, мм рт. ст.",
    "DBP, mmHg": "ДАТ, мм рт. ст.",
    "HR, bpm": "ЧСС, уд/хв",
})
export_table(t2, UA_DIR / "table_02_bp_by_context")
display(t2)
print("Exported table_02 (UA)")

In [ ]:
# --- Table 3: Variance decomposition (UA) ---
t3 = pd.read_csv(THESIS_DIR / "table_03_variance_decomposition_icc.csv")
t3["Variable"] = t3["Variable"].map(VAR_UA).fillna(t3["Variable"])
t3 = t3.rename(columns={
    "Variable": "Змінна",
    "Between-subject var": "Міжсуб\u2019єктна дисп.",
    "Within-subject var": "Внутрішньосуб\u2019єктна дисп.",
    "Total var": "Загальна дисп.",
    "Between %": "Міжсуб\u2019єктна %",
    "ICC CI low": "ICC ДІ нижн.",
    "ICC CI high": "ICC ДІ верхн.",
})
export_table(t3, UA_DIR / "table_03_variance_decomposition_icc")
display(t3)
print("Exported table_03 (UA)")

In [ ]:
# --- Table 4: Inferential results (UA) ---
t4 = pd.read_csv(THESIS_DIR / "table_04_inferential_effect_sizes.csv")
t4["Family"] = t4["Family"].map(FAMILY_UA).fillna(t4["Family"])
t4["Test"] = t4["Test"].map(TEST_UA).fillna(t4["Test"])
t4["Outcome"] = t4["Outcome"].map(VAR_UA).fillna(t4["Outcome"])
t4["Direction"] = t4["Direction"].map(DIR_UA).fillna(t4["Direction"])

# Translate comparison strings
def translate_comparison(s):
    for en, ua in CTX_UA.items():
        s = s.replace(en, ua)
    for en, ua in GROUP_UA.items():
        s = s.replace(en, ua)
    for en, ua in DIPPER_UA.items():
        s = s.replace(en, ua)
    s = s.replace(" vs ", " проти ")
    return s

t4["Comparison"] = t4["Comparison"].apply(translate_comparison)

t4 = t4.rename(columns={
    "Family": "Сім\u2019я тестів",
    "Comparison": "Порівняння",
    "Outcome": "Показник",
    "Test": "Критерій",
    "Statistic": "Статистика",
    "Effect_Size": "Розмір ефекту",
    "Direction": "Напрямок",
})
export_table(t4, UA_DIR / "table_04_inferential_effect_sizes")
print(f"Exported table_04 (UA): {len(t4)} rows")

In [ ]:
# --- Table 5: Dipping status (UA) ---
t5 = pd.read_csv(THESIS_DIR / "table_05_dipping_status.csv")
t5["Category"] = t5["Category"].map(DIPPER_UA).fillna(t5["Category"])
t5["Definition"] = t5["Category"].map(DIPPER_DEFS_UA).fillna(t5["Definition"])
t5 = t5.rename(columns={
    "Category": "Категорія",
    "Definition": "Визначення",
    "SBP n (%)": "САТ n (%)",
    "DBP n (%)": "ДАТ n (%)",
})
export_table(t5, UA_DIR / "table_05_dipping_status")
display(t5)
print("Exported table_05 (UA)")

In [ ]:
# --- Table 6: Task performance (UA) ---
t6 = pd.read_csv(THESIS_DIR / "table_06_task_performance.csv")
t6["Battery"] = t6["Battery"].str.replace("Battery", "Батарея")
t6 = t6.rename(columns={
    "Battery": "Батарея",
    "Stroop Effect": "Ефект Струпа",
    "Negative Stroop n (%)": "Негативний Струп n (%)",
    "Reaction Time": "Час реакції",
    "Missing": "Пропуски",
})
export_table(t6, UA_DIR / "table_06_task_performance")
display(t6)
print("Exported table_06 (UA)")

In [ ]:
# --- Table S1: Power analysis (UA) ---
tS1 = pd.read_csv(THESIS_DIR / "table_S1_power_analysis.csv")
tS1 = tS1.rename(columns={
    "Design": "Дизайн",
    "N (or N1 vs N2)": "N (або N1 проти N2)",
    "Min Detectable d (alpha=0.05, power=0.80)": "Мін. виявл. d (alpha=0,05; потужн.=0,80)",
    "Interpretation": "Інтерпретація",
})
interp_ua = {"medium": "середній", "large": "великий", "small": "малий"}
tS1["Інтерпретація"] = tS1["Інтерпретація"].map(interp_ua).fillna(tS1["Інтерпретація"])
design_ua = {
    "Paired (within-subject)": "Парний (внутрішньосуб\u2019єктний)",
    "Independent (Male vs Female)": "Незалежний (чол. проти жін.)",
    "Independent (Young vs Older)": "Незалежний (молоді проти старших)",
    "Independent (Healthy vs Diagnosed)": "Незалежний (здорові проти діагнозу)",
}
tS1["Дизайн"] = tS1["Дизайн"].map(design_ua).fillna(tS1["Дизайн"])
export_table(tS1, UA_DIR / "table_S1_power_analysis")
display(tS1)
print("Exported table_S1 (UA)")

In [ ]:
# --- Table S2: Correlations (UA) ---
tS2 = pd.read_csv(THESIS_DIR / "table_S2_correlation_results.csv")
pred_ua = {
    "age": "Вік", "bmi": "ІМТ", "bp_load_%": "Навант. АТ %",
    "sbp_dip_%": "Зниж. САТ %", "dbp_dip_%": "Зниж. ДАТ %",
    "sleep_duration_h": "Трив. сну (год)",
    "DBP_all_med": "Медіана ДАТ", "DBP_all_iqr": "IQR ДАТ",
}
outcome_corr_ua = {
    "DBP_Cognitive Task_DeltaBias": "ДАТ Когн. навант. DeltaBias",
    "DBP_Cognitive Task_Anomaly": "ДАТ Когн. навант. Anomaly",
    "DBP_Physical Task_DeltaBias": "ДАТ Фіз. навант. DeltaBias",
    "DBP_Physical Task_Anomaly": "ДАТ Фіз. навант. Anomaly",
    "DBP_Air Alert_DeltaBias": "ДАТ Повітр. тривога DeltaBias",
    "DBP_Air Alert_Anomaly": "ДАТ Повітр. тривога Anomaly",
}
tS2["Predictor"] = tS2["Predictor"].map(pred_ua).fillna(tS2["Predictor"])
tS2["Outcome"] = tS2["Outcome"].map(outcome_corr_ua).fillna(tS2["Outcome"])
tS2 = tS2.rename(columns={
    "Predictor": "Предиктор",
    "Outcome": "Показник",
    "Significant (FDR)": "Значущий (FDR)",
})
export_table(tS2, UA_DIR / "table_S2_correlation_results")
print(f"Exported table_S2 (UA): {len(tS2)} rows")

In [ ]:
# --- Table S3: Mixed model coefficients (UA) ---
tS3 = pd.read_csv(THESIS_DIR / "table_S3_mixedlm_coefficients.csv")
pred_lmm_ua = {
    "Intercept (Baseline)": "Перетин (Базовий рівень)",
    "Sleep": "Сон",
    "Cognitive Task": "Когнітивне навант.",
    "Physical Task": "Фізичне навант.",
}
tS3["Outcome"] = tS3["Outcome"].map(VAR_UA).fillna(tS3["Outcome"])
tS3["Predictor"] = tS3["Predictor"].map(pred_lmm_ua).fillna(tS3["Predictor"])
tS3 = tS3.rename(columns={
    "Outcome": "Показник",
    "Predictor": "Предиктор",
    "Coefficient": "Коефіцієнт",
    "95% CI": "95% ДІ",
})
export_table(tS3, UA_DIR / "table_S3_mixedlm_coefficients")
display(tS3)
print("Exported table_S3 (UA)")

In [ ]:
# --- Table S4: Sensitivity (UA) ---
tS4 = pd.read_csv(THESIS_DIR / "table_S4_sensitivity_extreme_exclusion.csv")
ds_ua = {"Original": "Оригінал", "Sensitivity (extremes removed)": "Без екстремальних значень"}
tS4["Dataset"] = tS4["Dataset"].map(ds_ua).fillna(tS4["Dataset"])
tS4 = tS4.rename(columns={
    "Dataset": "Набір даних",
    "N pairs": "N пар",
    "N non-zero diffs": "N ненульових різниць",
    "Median diff (Cog - BL)": "Медіана різниці (Когн. - Баз.)",
    "W statistic": "Статистика W",
    "p-value": "p-значення",
})
export_table(tS4, UA_DIR / "table_S4_sensitivity_extreme_exclusion")
display(tS4)
print("Exported table_S4 (UA)")

## Рисунки

In [ ]:
# --- Figure 1: Hemodynamic distributions (UA) ---
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, (col, ua_name) in zip(axes, [(Columns.SBP, "САТ"), (Columns.DBP, "ДАТ"), (Columns.HR, "ЧСС")]):
    vals = monitoring[col].dropna()
    ax.hist(vals, bins=50, density=True, alpha=0.6, color="steelblue", edgecolor="white")
    vals.plot.kde(ax=ax, color="navy", linewidth=1.5)
    ax.axvline(vals.median(), color="red", linestyle="--", linewidth=1.2, label=f"Медіана = {vals.median():.0f}")
    unit = UNITS_UA[ua_name]
    ax.set_xlabel(f"{ua_name} ({unit})")
    ax.set_ylabel("Щільність")
    ax.legend(fontsize=9)
plt.tight_layout()
fig.savefig(UA_DIR / "fig_01_hemodynamic_distributions.png", dpi=DPI, bbox_inches="tight")
plt.show()
plt.close(fig)
print("Saved fig_01 (UA)")

In [ ]:
# --- Figure 2: Circadian profiles (UA) ---
monitoring["hour"] = pd.to_datetime(monitoring[Columns.TIME]).dt.hour
hourly = monitoring.groupby("hour")[[Columns.SBP, Columns.DBP, Columns.HR]].agg(["mean", "sem"])

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, (col, ua_name) in zip(axes, [(Columns.SBP, "САТ"), (Columns.DBP, "ДАТ"), (Columns.HR, "ЧСС")]):
    means = hourly[(col, "mean")]
    sems = hourly[(col, "sem")]
    ax.plot(means.index, means.values, "o-", color="steelblue", linewidth=2, markersize=4)
    ax.fill_between(means.index, means - sems, means + sems, alpha=0.2, color="steelblue")
    ax.axvspan(23, 24, alpha=0.08, color="gray")
    ax.axvspan(0, 6, alpha=0.08, color="gray", label="Типовий сон")
    unit = UNITS_UA[ua_name]
    ax.set_xlabel("Година доби")
    ax.set_ylabel(f"{ua_name} ({unit})")
    ax.set_title(f"Циркадний профіль {ua_name}")
    ax.set_xticks(range(0, 24, 3))
    ax.legend(fontsize=8)
plt.tight_layout()
fig.savefig(UA_DIR / "fig_02_circadian_profiles.png", dpi=DPI, bbox_inches="tight")
plt.show()
plt.close(fig)
print("Saved fig_02 (UA)")

In [ ]:
# --- Figure 3: Sleep transition profiles (UA) ---
SLEEP_CONTEXTS = ["Before Testing", "Sleep", "Baseline"]
SLEEP_CTX_UA = ["Перед сном", "Сон", "Після сну"]
sleep_data = ctx_medians[ctx_medians[Columns.LABEL].isin(SLEEP_CONTEXTS)].copy()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (med_col, ua_name) in zip(axes, [("SBP_med", "САТ"), ("DBP_med", "ДАТ"), ("HR_med", "ЧСС")]):
    for pid in sleep_data["participant_id"].unique():
        subj = sleep_data[sleep_data["participant_id"] == pid]
        vals = []
        for ctx in SLEEP_CONTEXTS:
            row = subj[subj[Columns.LABEL] == ctx]
            vals.append(row[med_col].values[0] if len(row) > 0 else np.nan)
        ax.plot(range(3), vals, "o-", color="steelblue", alpha=0.3, linewidth=0.8, markersize=3)
    # Group mean
    group_means = []
    for ctx in SLEEP_CONTEXTS:
        group_means.append(sleep_data[sleep_data[Columns.LABEL] == ctx][med_col].mean())
    ax.plot(range(3), group_means, "o-", color="darkred", linewidth=3, markersize=8, label="Середнє групи")
    unit = UNITS_UA[ua_name]
    ax.set_ylabel(f"{ua_name} ({unit})")
    ax.set_xticks(range(3))
    ax.set_xticklabels(SLEEP_CTX_UA, fontsize=9)
    ax.set_title(f"{ua_name}: перехід сон\u2013неспання")
    ax.legend(fontsize=8)
plt.tight_layout()
fig.savefig(UA_DIR / "fig_03_sleep_transition_profiles.png", dpi=DPI, bbox_inches="tight")
plt.show()
plt.close(fig)
print("Saved fig_03 (UA)")

In [ ]:
# --- Figure 4: Spaghetti subject trajectories (UA) ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (col, ua_name) in zip(axes, [(Columns.SBP, "САТ"), (Columns.DBP, "ДАТ"), (Columns.HR, "ЧСС")]):
    for pid, grp in monitoring.groupby(Columns.PAT_ID):
        ax.plot(pd.to_datetime(grp[Columns.TIME]).dt.hour + pd.to_datetime(grp[Columns.TIME]).dt.minute / 60,
                grp[col], alpha=0.25, linewidth=0.6)
    unit = UNITS_UA[ua_name]
    ax.set_xlabel("Година доби")
    ax.set_ylabel(f"{ua_name} ({unit})")
    ax.set_title(f"Індивідуальні траєкторії {ua_name}")
    ax.set_xticks(range(0, 24, 3))
plt.tight_layout()
fig.savefig(UA_DIR / "fig_04_spaghetti_subject_trajectories.png", dpi=DPI, bbox_inches="tight")
plt.show()
plt.close(fig)
print("Saved fig_04 (UA)")

In [ ]:
# --- Figure 5: Subject correlation heatmap (UA) ---
corr_cols = [
    "age", "bmi", "bp_load_%", "sbp_dip_%", "dbp_dip_%", "sleep_duration_h",
    "SBP_all_med", "DBP_all_med", "HR_all_med",
    "SBP_all_iqr", "DBP_all_iqr", "HR_all_iqr",
]
corr_labels_ua = [
    "Вік", "ІМТ", "Навант. АТ %", "Зниж. САТ %", "Зниж. ДАТ %", "Трив. сну",
    "Мед. САТ", "Мед. ДАТ", "Мед. ЧСС",
    "IQR САТ", "IQR ДАТ", "IQR ЧСС",
]
corr_matrix = agg[corr_cols].corr(method="spearman")

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    corr_matrix, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
    xticklabels=corr_labels_ua, yticklabels=corr_labels_ua,
    ax=ax, square=True, linewidths=0.5,
    cbar_kws={"label": "Кореляція Спірмена"},
)
ax.set_title("Кореляційна матриця (рівень учасника)", fontsize=13)
plt.tight_layout()
fig.savefig(UA_DIR / "fig_05_subject_correlation_heatmaps.png", dpi=DPI, bbox_inches="tight")
plt.show()
plt.close(fig)
print("Saved fig_05 (UA)")

In [ ]:
# --- Figure 6: Context boxplots (UA) ---
CONTEXT_ORDER = [
    Columns.LABEL_BASELINE, Columns.LABEL_SLEEP, Columns.LABEL_BEFORE_TESTING,
    Columns.LABEL_COGNITIVE_PRE, Columns.LABEL_COGNITIVE_TASK,
    Columns.LABEL_PHYSICAL_PRE, Columns.LABEL_PHYSICAL_TASK,
    Columns.LABEL_BREAK, Columns.LABEL_AIR_ALERT,
]
plot_ctx = ctx_medians[ctx_medians[Columns.LABEL].isin(CONTEXT_ORDER)].copy()
plot_ctx["label_ua"] = plot_ctx[Columns.LABEL].map(CTX_UA)
ctx_order_ua = [CTX_UA[c] for c in CONTEXT_ORDER]
plot_ctx["label_ua"] = pd.Categorical(plot_ctx["label_ua"], categories=ctx_order_ua, ordered=True)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
measures = [("SBP_med", "САТ (мм рт. ст.)"), ("DBP_med", "ДАТ (мм рт. ст.)"), ("HR_med", "ЧСС (уд/хв)")]
for ax, (col, ylabel) in zip(axes, measures):
    sns.boxplot(data=plot_ctx, x="label_ua", y=col, ax=ax, palette="Set2", fliersize=3)
    sns.stripplot(data=plot_ctx, x="label_ua", y=col, ax=ax, color="black", size=3, alpha=0.4, jitter=0.2)
    ax.set_ylabel(ylabel)
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=45, labelsize=7)
fig.suptitle("Медіана АТ/ЧСС на рівні учасника за контекстами моніторування", fontsize=14, y=1.02)
plt.tight_layout()
fig.savefig(UA_DIR / "fig_06_context_boxplots.png", dpi=DPI, bbox_inches="tight")
plt.show()
plt.close(fig)
print("Saved fig_06 (UA)")

In [ ]:
# --- Figure 7: Effect size forest plot (UA) ---
t4_en = pd.read_csv(THESIS_DIR / "table_04_inferential_effect_sizes.csv")
paired = t4_en[t4_en["Family"] == "Paired (Wilcoxon)"].copy()

# Parse CI
paired["CI_low"] = paired["CI_95"].str.extract(r'\[([\-0-9.]+)').astype(float)
paired["CI_high"] = paired["CI_95"].str.extract(r',\s*([\-0-9.]+)\]').astype(float)
paired["r_rb"] = paired["Effect_Size"].str.extract(r'=\s*([\-0-9.]+)').astype(float)

# Translate labels
def ua_label(row):
    comp = row["Comparison"]
    for en, ua in CTX_UA.items():
        comp = comp.replace(en, ua)
    comp = comp.replace(" vs ", " проти ")
    out = VAR_UA.get(row["Outcome"], row["Outcome"])
    return f"{comp} [{out}]"

paired["label"] = paired.apply(ua_label, axis=1)
plot_df = paired.sort_values("r_rb").reset_index(drop=True)

outcome_colors = {"SBP": "#e74c3c", "DBP": "#3498db", "HR": "#2ecc71"}
fig, ax = plt.subplots(figsize=(10, max(4, len(plot_df) * 0.45)))
for i, (_, row) in enumerate(plot_df.iterrows()):
    color = outcome_colors[row["Outcome"]]
    err_low = max(0, row["r_rb"] - row["CI_low"])
    err_high = max(0, row["CI_high"] - row["r_rb"])
    ax.errorbar(row["r_rb"], i, xerr=[[err_low], [err_high]],
                fmt="o", color=color, ecolor=color, elinewidth=2,
                capsize=4, markersize=7, markeredgecolor="white", markeredgewidth=0.5)

ax.axvline(0, color="black", linestyle="--", linewidth=1, alpha=0.7)
ax.set_yticks(range(len(plot_df)))
ax.set_yticklabels(plot_df["label"], fontsize=9)
ax.set_xlabel("Рангово-бісеріальна кореляція ($r_{rb}$)")
ax.set_title("Розміри ефектів: парні внутрішньосуб\u2019єктні порівняння")

from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor=c, markersize=8, label=VAR_UA[k])
    for k, c in outcome_colors.items()
]
ax.legend(handles=legend_elements, loc="lower right", fontsize=9)
plt.tight_layout()
fig.savefig(UA_DIR / "fig_07_effect_size_forest.png", dpi=DPI, bbox_inches="tight")
plt.show()
plt.close(fig)
print("Saved fig_07 (UA)")

In [ ]:
# --- Figure 8: Task performance distributions (UA) ---
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

for i, batt in enumerate([1, 2]):
    stroop_col = f"stroop_effect_battery_{batt}"
    rt_col = f"avg_reaction_time_battery_{batt}"

    ax_s = axes[0, i]
    vals_s = agg[stroop_col].dropna()
    ax_s.hist(vals_s, bins=15, density=True, alpha=0.6, color="steelblue", edgecolor="white")
    if len(vals_s) > 2:
        vals_s.plot.kde(ax=ax_s, color="navy", linewidth=1.5)
    ax_s.set_xlabel("Ефект Струпа (мс)")
    ax_s.set_ylabel("Щільність")
    ax_s.set_title(f"Батарея {batt}: Ефект Струпа")

    ax_r = axes[1, i]
    vals_r = agg[rt_col].dropna()
    ax_r.hist(vals_r, bins=15, density=True, alpha=0.6, color="coral", edgecolor="white")
    if len(vals_r) > 2:
        vals_r.plot.kde(ax=ax_r, color="darkred", linewidth=1.5)
    ax_r.set_xlabel("Час реакції (с)")
    ax_r.set_ylabel("Щільність")
    ax_r.set_title(f"Батарея {batt}: Час реакції")

plt.tight_layout()
fig.savefig(UA_DIR / "fig_08_task_performance_distributions.png", dpi=DPI, bbox_inches="tight")
plt.show()
plt.close(fig)
print("Saved fig_08 (UA)")

## Додаткові рисунки (S1–S4)

In [ ]:
# --- Figure S1: Q-Q plots (UA) ---
qq_vars = [
    ("age", "Вік"), ("bmi", "ІМТ"), ("bp_load_%", "Навант. АТ %"),
    ("sbp_dip_%", "Зниж. САТ %"), ("dbp_dip_%", "Зниж. ДАТ %"),
    ("sleep_duration_h", "Трив. сну"),
    ("SBP_all_med", "Мед. САТ"), ("DBP_all_med", "Мед. ДАТ"), ("HR_all_med", "Мед. ЧСС"),
]
n_vars = len(qq_vars)
ncols = 3
nrows = (n_vars + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(15, 4 * nrows))
axes_flat = axes.flatten()

for idx, (col, ua_name) in enumerate(qq_vars):
    ax = axes_flat[idx]
    vals = agg[col].dropna().values
    stats.probplot(vals, dist="norm", plot=ax)
    ax.set_title(ua_name)
    ax.set_xlabel("Теоретичні квантилі")
    ax.set_ylabel("Емпіричні квантилі")

for idx in range(len(qq_vars), len(axes_flat)):
    axes_flat[idx].set_visible(False)

fig.suptitle("Q-Q графіки: оцінка нормальності розподілу", fontsize=14, y=1.01)
plt.tight_layout()
fig.savefig(UA_DIR / "fig_S1_qq_plots.png", dpi=DPI, bbox_inches="tight")
plt.show()
plt.close(fig)
print("Saved fig_S1 (UA)")

In [ ]:
# --- Figure S2: Power curves (UA) ---
from statsmodels.stats.power import TTestPower, TTestIndPower

effect_sizes = np.linspace(0.05, 1.5, 100)
paired_power = TTestPower()
indep_power = TTestIndPower()

power_28 = [paired_power.solve_power(effect_size=d, nobs=28, alpha=0.05) for d in effect_sizes]
power_11 = [paired_power.solve_power(effect_size=d, nobs=11, alpha=0.05) for d in effect_sizes]
power_16v12 = [indep_power.solve_power(effect_size=d, nobs1=16, ratio=12/16, alpha=0.05) for d in effect_sizes]

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(effect_sizes, power_28, label="Парний N=28 (повна когорта)", linewidth=2)
ax.plot(effect_sizes, power_11, label="Парний N=11 (повітряна тривога)", linewidth=2, linestyle="--")
ax.plot(effect_sizes, power_16v12, label="Незалежний 16 проти 12 (чол. проти жін.)", linewidth=2, linestyle=":")
ax.axhline(0.80, color="gray", linestyle="-.", alpha=0.7, label="Потужність = 0,80")

for d_val, label in [(0.2, "малий"), (0.5, "середній"), (0.8, "великий")]:
    ax.axvline(d_val, color="lightgray", linestyle=":", alpha=0.5)
    ax.text(d_val + 0.02, 0.05, label, fontsize=8, color="gray")

ax.set_xlabel("Розмір ефекту (d Коена / $d_z$)")
ax.set_ylabel("Статистична потужність")
ax.set_title("Апостеріорний аналіз потужності")
ax.legend(fontsize=9)
ax.set_xlim(0, 1.5)
ax.set_ylim(0, 1.05)
plt.tight_layout()
fig.savefig(UA_DIR / "fig_S2_power_curves.png", dpi=DPI, bbox_inches="tight")
plt.show()
plt.close(fig)
print("Saved fig_S2 (UA)")

In [ ]:
# --- Figures S3 & S4: Mixed model (UA) ---
# These require re-fitting the mixed models to extract marginal means and residuals
import statsmodels.formula.api as smf
from analysis.thesis_stats import add_sin_cos_hour

mon_model = monitoring.copy()
mon_model = mon_model[mon_model[Columns.LABEL].isin([
    Columns.LABEL_BASELINE, Columns.LABEL_SLEEP,
    Columns.LABEL_COGNITIVE_TASK, Columns.LABEL_PHYSICAL_TASK,
])].dropna(subset=[Columns.SBP, Columns.DBP, Columns.HR])
mon_model = add_sin_cos_hour(mon_model, Columns.TIME)
mon_model["context"] = pd.Categorical(
    mon_model[Columns.LABEL],
    categories=[Columns.LABEL_BASELINE, Columns.LABEL_SLEEP,
                Columns.LABEL_COGNITIVE_TASK, Columns.LABEL_PHYSICAL_TASK],
)

ctx_labels_ua_short = {
    Columns.LABEL_BASELINE: "Базовий\nрівень",
    Columns.LABEL_SLEEP: "Сон",
    Columns.LABEL_COGNITIVE_TASK: "Когнітивне\nнавант.",
    Columns.LABEL_PHYSICAL_TASK: "Фізичне\nнавант.",
}

# Fit models and compute marginal means
fig_mm, axes_mm = plt.subplots(1, 3, figsize=(15, 5))
dbp_model = None

for ax, (outcome, ua_name) in zip(axes_mm, [(Columns.DBP, "ДАТ"), (Columns.SBP, "САТ"), (Columns.HR, "ЧСС")]):
    formula = f"{outcome} ~ C(context, Treatment(reference='{Columns.LABEL_BASELINE}')) + sin_hour + cos_hour"
    try:
        md = smf.mixedlm(formula, mon_model, groups=mon_model[Columns.PAT_ID])
        mdf = md.fit(reml=True, method="lbfgs")
        if outcome == Columns.DBP:
            dbp_model = mdf
    except Exception as e:
        print(f"  Model for {ua_name} failed: {e}")
        continue

    # Marginal means: intercept + context coefficient (sin/cos at 0)
    intercept = mdf.fe_params["Intercept"]
    ctx_list = [Columns.LABEL_BASELINE, Columns.LABEL_SLEEP,
                Columns.LABEL_COGNITIVE_TASK, Columns.LABEL_PHYSICAL_TASK]
    means = []
    cis_low = []
    cis_high = []
    for ctx in ctx_list:
        if ctx == Columns.LABEL_BASELINE:
            means.append(intercept)
            cis_low.append(mdf.conf_int().loc["Intercept", 0])
            cis_high.append(mdf.conf_int().loc["Intercept", 1])
        else:
            param_name = [p for p in mdf.fe_params.index if ctx in p]
            if param_name:
                coef = mdf.fe_params[param_name[0]]
                means.append(intercept + coef)
                ci = mdf.conf_int().loc[param_name[0]]
                cis_low.append(intercept + ci[0])
                cis_high.append(intercept + ci[1])
            else:
                means.append(np.nan)
                cis_low.append(np.nan)
                cis_high.append(np.nan)

    x_pos = range(len(ctx_list))
    errors = [[m - lo for m, lo in zip(means, cis_low)],
              [hi - m for m, hi in zip(means, cis_high)]]
    ax.errorbar(x_pos, means, yerr=errors, fmt="o-", capsize=5, markersize=8, linewidth=2)
    unit = UNITS_UA[ua_name]
    ax.set_ylabel(f"{ua_name} ({unit})")
    ax.set_xticks(list(x_pos))
    ax.set_xticklabels([ctx_labels_ua_short[c] for c in ctx_list], fontsize=8)
    ax.set_title(f"{ua_name} \u2014 маргінальні середні")

plt.tight_layout()
fig_mm.savefig(UA_DIR / "fig_S3_mixed_model_marginal_means.png", dpi=DPI, bbox_inches="tight")
plt.show()
plt.close(fig_mm)
print("Saved fig_S3 (UA)")

# --- Figure S4: DBP model diagnostics ---
if dbp_model is not None:
    resid = dbp_model.resid
    fitted = dbp_model.fittedvalues

    fig_diag, axes_d = plt.subplots(1, 3, figsize=(15, 4.5))

    axes_d[0].scatter(fitted, resid, alpha=0.3, s=10)
    axes_d[0].axhline(0, color="red", linestyle="--")
    axes_d[0].set_xlabel("Прогнозовані значення")
    axes_d[0].set_ylabel("Залишки")
    axes_d[0].set_title("Залишки проти прогнозованих \u2014 модель ДАТ")

    stats.probplot(resid, dist="norm", plot=axes_d[1])
    axes_d[1].set_title("Q-Q графік залишків моделі ДАТ")
    axes_d[1].set_xlabel("Теоретичні квантилі")
    axes_d[1].set_ylabel("Емпіричні квантилі")

    axes_d[2].hist(resid, bins=40, density=True, alpha=0.6, color="steelblue", edgecolor="white")
    axes_d[2].set_xlabel("Залишки")
    axes_d[2].set_ylabel("Щільність")
    axes_d[2].set_title("Розподіл залишків моделі ДАТ")

    plt.tight_layout()
    fig_diag.savefig(UA_DIR / "fig_S4_dbp_model_diagnostics.png", dpi=DPI, bbox_inches="tight")
    plt.show()
    plt.close(fig_diag)
    print("Saved fig_S4 (UA)")
else:
    print("DBP model not available, skipping fig_S4")

## Підсумок

In [ ]:
import os

expected = [
    # Tables
    "table_01_cohort_demographics.csv", "table_01_cohort_demographics.tex",
    "table_02_bp_by_context.csv", "table_02_bp_by_context.tex",
    "table_03_variance_decomposition_icc.csv", "table_03_variance_decomposition_icc.tex",
    "table_04_inferential_effect_sizes.csv", "table_04_inferential_effect_sizes.tex",
    "table_05_dipping_status.csv", "table_05_dipping_status.tex",
    "table_06_task_performance.csv", "table_06_task_performance.tex",
    "table_S1_power_analysis.csv", "table_S1_power_analysis.tex",
    "table_S2_correlation_results.csv", "table_S2_correlation_results.tex",
    "table_S3_mixedlm_coefficients.csv", "table_S3_mixedlm_coefficients.tex",
    "table_S4_sensitivity_extreme_exclusion.csv", "table_S4_sensitivity_extreme_exclusion.tex",
    # Figures
    "fig_01_hemodynamic_distributions.png",
    "fig_02_circadian_profiles.png",
    "fig_03_sleep_transition_profiles.png",
    "fig_04_spaghetti_subject_trajectories.png",
    "fig_05_subject_correlation_heatmaps.png",
    "fig_06_context_boxplots.png",
    "fig_07_effect_size_forest.png",
    "fig_08_task_performance_distributions.png",
    "fig_S1_qq_plots.png",
    "fig_S2_power_curves.png",
    "fig_S3_mixed_model_marginal_means.png",
    "fig_S4_dbp_model_diagnostics.png",
]

print("Українські артефакти:")
for name in expected:
    path = UA_DIR / name
    if path.exists():
        size = os.path.getsize(path)
        print(f"  [OK] {name} ({size:,} байт)")
    else:
        print(f"  [ВІДСУТНІЙ] {name}")

print(f"\nЗагалом: {sum(1 for n in expected if (UA_DIR / n).exists())}/{len(expected)}")